# NB02 — Capture & the Virtual Pipeline
真實系統入面，相機要同投影儀**同步**：每播一張圖案影一張相。
工業方案用硬件觸發線（µs 級）；我哋嘅 universal pipeline 用**時間協議**：
每張圖案播 ≥16.7ms（60Hz refresh floor），相機連續影，事後對齊。
呢課先用 repo 入面嘅真實捕捉數據集做實驗，再用 virtual 模式模擬成個 loop。

In [1]:
import sys, pathlib
# repo-root relative imports so the notebook runs from anywhere
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / 'edu' / 'sl_edu').exists()) \
       if not (pathlib.Path.cwd() / 'edu' / 'sl_edu').exists() else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'edu'))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 4)
DATA = ROOT / 'data'


## 1. 真實數據集：`data/shiftGraycode`
呢 9 張相係真相機影真場景（4 相位 + 5 格雷），同 C++ test 用嘅係同一批。

In [2]:
from sl_edu import oracle

imgs = oracle.load_shift_graycode(ROOT)
print(f"{len(imgs)} images, {imgs[0].shape[1]}x{imgs[0].shape[0]}")

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for k, ax in enumerate(axes.flat):
    ax.imshow(imgs[k], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'img {k}' + (' (phase)' if k < 4 else f' (gray bit {k-4})'))
    ax.axis('off')
plt.tight_layout(); plt.show()

9 images, 1280x1024


/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82683/3269004373.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


留意：相位圖（頭 4 張）睇落似漸變斜坡，格雷圖（後 5 張）係黑白塊。
場景入面嘅物體令條紋變形——**變形量就係深度資訊**。

## 2. 信心圖（confidence map）
邊啲 pixel 信得過？SLMaster 用 4 張相位圖嘅**平均亮度**做信心：
太暗（陰影/黑面）或太光（反光飽和）嘅 pixel 喺 decode 時會被丟棄。

In [3]:
from sl_edu import decode

conf = decode.confidence_map(imgs[:4])
plt.imshow(conf, cmap='hot'); plt.colorbar(label='mean intensity')
plt.title('confidence map'); plt.show()
print(f'pixels above threshold 5: {np.mean(conf > 5):.1%}')

pixels above threshold 5: 100.0%


/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82683/3024297858.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('confidence map'); plt.show()


## 3. Virtual projector loop（冇硬件都玩到）
C++ Phase 3 加咗 `SLMASTER_MONITOR_VIRTUAL=1`；Python 版一樣有 virtual 模式。
下面個 projector 真係逐張「播」（只係冇 window），timing 行真實 refresh floor。

In [4]:
import time
from sl_edu import backends, patterns

pats = patterns.generate(640, 480, shift_time=4, n_periods=8)
proj = backends.PyMonitorProjector(exposure_ms=20, virtual=True)
proj.set_patterns(pats)
proj.project()

seen = []
t0 = time.time()
while time.time() - t0 < 0.35:
    if proj.current_pattern not in seen:
        seen.append(proj.current_pattern)
    time.sleep(0.005)
proj.stop()
print('patterns displayed in order:', seen)
print('-> 成個序列循環播，每張 ≥16.7ms，直到 pause/stop')

patterns displayed in order: [0, 1, 2]
-> 成個序列循環播，每張 ≥16.7ms，直到 pause/stop


## 4. 相機端：image-sequence 後端
我哋嘅 `PyOpenCvCamera` 支援 `dir/%d.bmp` 模式——將數據集當成「相機」重播。

In [5]:
cam = backends.PyOpenCvCamera(str(DATA / 'shiftGraycode' / '%d.bmp'))
assert cam.open(), 'sequence open failed'
cam.start()
frames = cam.capture(9, timeout_s=10)
cam.stop()
print(f'captured {len(frames)} frames via the camera backend')
print('frame 0 identical to direct load:',
      np.array_equal(frames[0], imgs[0]))

captured 9 frames via the camera backend
frame 0 identical to direct load: True


下一課：decode——由 9 張灰階圖還原每個 pixel 嘅絕對相位。